In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    """Find the repository root from Jupyter's current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from inside "
        "NYC_Healthcare_Accessibility."
    )


REPO_ROOT = find_repository_root()
INPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_inputs"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


In [3]:

df = pd.read_csv(INPUT_DIR / "NEW_YORK_CITY_ALL_INDICATORS_6.csv")
df.head(6500)

,Unnamed: 0,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,1,360610002011,o_360610002011,1294.883739,1067.296,0,19.750000,18.116667,2.100000
1,2,360610002012,o_360610002012,1887.846508,1125.124,0,24.833333,19.083333,2.050000
2,3,360610002021,o_360610002021,2267.091673,1076.668,0,23.033333,18.250000,2.316667
3,4,360610002022,o_360610002022,2477.358670,1112.161,0,24.200000,18.850000,1.150000
4,5,360610002023,o_360610002023,2339.396673,1148.973,0,24.283333,19.500000,1.066667
...,...,...,...,...,...,...,...,...,...
6495,6496,360811451022,o_360811451022,3582.486358,486.392,0,19.616667,8.266667,2.166667
6496,6497,360811459001,o_360811459001,2486.686084,1508.068,0,29.616667,25.616667,3.450000
6497,6498,360811459002,o_360811459002,2214.762084,1236.144,0,25.100000,21.100000,2.966667
6498,6499,360811459003,o_360811459003,2162.428084,1183.810,0,24.266667,20.266667,3.800000


In [4]:
print(df.shape)
print(df.columns.tolist())

(6569, 9)
['Unnamed: 0', 'GEOID_TEXT', 'from_id', 'total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [5]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["GEOID_TEXT"] = df["from_id"].astype(str).str.replace("o_", "", regex=False)

df[["from_id", "GEOID_TEXT"]].head(343)

,from_id,GEOID_TEXT
0,o_360610002011,360610002011
1,o_360610002012,360610002012
2,o_360610002021,360610002021
3,o_360610002022,360610002022
4,o_360610002023,360610002023
...,...,...
338,o_360610083002,360610083002
339,o_360610083003,360610083003
340,o_360610084001,360610084001
341,o_360610084002,360610084002


In [7]:
benefit_cols = []
cost_cols = ["total_distance",

    "walking_distance",

    "transfers",

    "travel_time_total",

    "walking_time",

    "wait_time_total"]

criteria_cols = benefit_cols + cost_cols

print("Benefit indicators, higher is better:")
print(benefit_cols)

print("\nCost indicators, lower is better:")
print(cost_cols)

print("\nAll criteria:")
print(criteria_cols)

Benefit indicators, higher is better:
[]

Cost indicators, lower is better:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']

All criteria:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total']


In [8]:
min_max_table = pd.DataFrame({
    "min": df[criteria_cols].min(),
    "max": df[criteria_cols].max()
})

print("Min and max for each indicator:")
display(min_max_table)

Min and max for each indicator:


,min,max
total_distance,307.234615,26862.592080
walking_distance,108.267000,5403.717000
transfers,0.000000,2.000000
travel_time_total,1.000000,120.800000
walking_time,1.783333,90.983333
wait_time_total,1.016667,35.016667


In [9]:
# All of our current indicators are cost indicators
# Lower = better, so we use: (max - value) / (max - min)

normalized = pd.DataFrame(index=df.index)

for col in criteria_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    
    normalized[col] = (max_val - df[col]) / (max_val - min_val)

print("Normalized values:")
display(normalized.head())

Normalized values:


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.962808,0.818896,1.0,0.843489,0.816891,0.968137
1,0.940479,0.807975,1.0,0.801057,0.806054,0.969608
2,0.926197,0.817126,1.0,0.816082,0.815396,0.961765
3,0.918279,0.810423,1.0,0.806344,0.808670,0.996078
4,0.923474,0.803472,1.0,0.805648,0.801383,0.998529


In [10]:
r_column_sums = normalized[criteria_cols].sum()

print("Step 2 preparation: Sum of each standardized column")
print("These sums go in the denominator for p_ij.")
display(r_column_sums)

Step 2 preparation: Sum of each standardized column
These sums go in the denominator for p_ij.


total_distance       5888.729178
walking_distance     5118.715542
transfers            5807.000000
travel_time_total    5098.035754
walking_time         5106.990658
wait_time_total      5961.088726
dtype: float64

In [11]:
P = normalized[criteria_cols] / r_column_sums

print("Step 2: Probability matrix p_ij")
print("Each standardized value is divided by its column total.")
display(P.head())

Step 2: Probability matrix p_ij
Each standardized value is divided by its column total.


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000164,0.000160,0.000172,0.000165,0.000160,0.000162
1,0.000160,0.000158,0.000172,0.000157,0.000158,0.000163
2,0.000157,0.000160,0.000172,0.000160,0.000160,0.000161
3,0.000156,0.000158,0.000172,0.000158,0.000158,0.000167
4,0.000157,0.000157,0.000172,0.000158,0.000157,0.000168


In [12]:
print("values should add up to one for each column, since they are probabilities.")
display(P.sum())

values should add up to one for each column, since they are probabilities.


total_distance       1.0
walking_distance     1.0
transfers            1.0
travel_time_total    1.0
walking_time         1.0
wait_time_total      1.0
dtype: float64

In [13]:
P_safe = P.replace(0, 1e-12)

print("Step 3 preparation: Replace 0 values so ln(0) does not break")
display(P_safe.head())

Step 3 preparation: Replace 0 values so ln(0) does not break


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,0.000164,0.000160,0.000172,0.000165,0.000160,0.000162
1,0.000160,0.000158,0.000172,0.000157,0.000158,0.000163
2,0.000157,0.000160,0.000172,0.000160,0.000160,0.000161
3,0.000156,0.000158,0.000172,0.000158,0.000158,0.000167
4,0.000157,0.000157,0.000172,0.000158,0.000157,0.000168


In [14]:
ln_P = np.log(P_safe)

print("Step 3: Natural log of p_ij")
display(ln_P.head())

Step 3: Natural log of p_ij


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-8.718697,-8.740457,-8.666819,-8.706819,-8.740615,-8.725390
1,-8.742162,-8.753883,-8.666819,-8.758433,-8.753970,-8.723872
2,-8.757463,-8.742621,-8.666819,-8.739851,-8.742447,-8.731994
3,-8.766049,-8.750857,-8.666819,-8.751856,-8.750730,-8.696938
4,-8.760408,-8.759472,-8.666819,-8.752719,-8.759782,-8.694480


In [15]:
P_ln_P = P_safe * ln_P

print("Step 4: p_ij times ln(p_ij)")
display(P_ln_P.head(8))

Step 4: p_ij times ln(p_ij)


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total
0,-0.001426,-0.001398,-0.001492,-0.001441,-0.001398,-0.001417
1,-0.001396,-0.001382,-0.001492,-0.001376,-0.001382,-0.001419
2,-0.001377,-0.001396,-0.001492,-0.001399,-0.001396,-0.001409
3,-0.001367,-0.001385,-0.001492,-0.001384,-0.001386,-0.001453
4,-0.001374,-0.001375,-0.001492,-0.001383,-0.001375,-0.001456
5,-0.001362,-0.001461,-0.001492,-0.001425,-0.001461,-0.001330
6,-0.001322,-0.001310,-0.001492,-0.001352,-0.001311,-0.001277
7,-0.001350,-0.001467,-0.000806,-0.001412,-0.001467,-0.001331


In [16]:
p_ln_p_sums = P_ln_P.sum(axis=0)

print("Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator")
display(p_ln_p_sums)

Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator


total_distance      -8.783442
walking_distance    -8.774918
transfers           -8.745600
travel_time_total   -8.776452
walking_time        -8.774718
wait_time_total     -8.784779
dtype: float64

In [17]:
# Step 5: Calculate entropy for each indicator

n = len(normalized)
k = 1 / np.log(n)

entropy = -k * p_ln_p_sums

print("Number of rows:", n)
print("k value:", k)
print("Step 5: Entropy for each indicator")
display(entropy)

Number of rows: 6569
k value: 0.11376412989554004
Step 5: Entropy for each indicator


total_distance       0.999241
walking_distance     0.998271
transfers            0.994936
travel_time_total    0.998445
walking_time         0.998248
wait_time_total      0.999393
dtype: float64

In [30]:
# df["fare"].value_counts()

In [18]:
# Step 6: Diversity
# Diversity tells us how much useful variation each indicator has

diversity = 1 - entropy

print("Step 6: Diversity for each indicator")
display(diversity)

Step 6: Diversity for each indicator


total_distance       0.000759
walking_distance     0.001729
transfers            0.005064
travel_time_total    0.001555
walking_time         0.001752
wait_time_total      0.000607
dtype: float64

In [19]:
# Step 7: Calculate entropy weights
# Weight = diversity of one indicator / total diversity of all indicators

weights = diversity / diversity.sum()

print("Step 7: Entropy weights for each indicator")
display(weights)

Step 7: Entropy weights for each indicator


total_distance       0.066226
walking_distance     0.150798
transfers            0.441671
travel_time_total    0.135570
walking_time         0.152778
wait_time_total      0.052956
dtype: float64

In [20]:
weights_table = pd.DataFrame({
    "entropy": entropy,
    "diversity": diversity,
    "weight": weights
})

print("Final entropy weight table:")
display(weights_table)

Final entropy weight table:


,entropy,diversity,weight
total_distance,0.999241,0.000759,0.066226
walking_distance,0.998271,0.001729,0.150798
transfers,0.994936,0.005064,0.441671
travel_time_total,0.998445,0.001555,0.135570
walking_time,0.998248,0.001752,0.152778
wait_time_total,0.999393,0.000607,0.052956


In [21]:
display(weights_table.sort_values(by="weight", ascending=False))

,entropy,diversity,weight
transfers,0.994936,0.005064,0.441671
walking_time,0.998248,0.001752,0.152778
walking_distance,0.998271,0.001729,0.150798
travel_time_total,0.998445,0.001555,0.135570
total_distance,0.999241,0.000759,0.066226
wait_time_total,0.999393,0.000607,0.052956


In [22]:
# Step 8: Calculate final EWM accessibility score
# Formula: score for each block group = sum(normalized value * indicator weight)

df["ewm_accessibility_score"] = (normalized[criteria_cols] * weights).sum(axis=1)

print("Step 8: Final EWM accessibility score")
display(df[["from_id", "GEOID_TEXT", "ewm_accessibility_score"]].head(30))

Step 8: Final EWM accessibility score


,from_id,GEOID_TEXT,ewm_accessibility_score
0,o_360610002011,360610002011,0.919346
1,o_360610002012,360610002012,0.908890
2,o_360610002021,360610002021,0.912373
3,o_360610002022,360610002022,0.910307
4,o_360610002023,360610002023,0.908525
5,o_360610002024,360610002024,0.923853
6,o_360610002025,360610002025,0.883055
7,o_360610002026,360610002026,0.702480
8,o_360610005001,360610005001,0.645979
9,o_360610005002,360610005002,0.840455


In [23]:
print("Score summary:")
display(df["ewm_accessibility_score"].describe())

Score summary:


count    6569.000000
mean        0.839355
std         0.128553
min         0.067113
25%         0.780993
50%         0.889356
75%         0.928272
max         0.992150
Name: ewm_accessibility_score, dtype: float64

In [24]:
final_results = df[
    ["GEOID_TEXT", "from_id"] + criteria_cols + ["ewm_accessibility_score"]
].copy()

display(final_results.head(20))

,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,ewm_accessibility_score
0,360610002011,o_360610002011,1294.883739,1067.296,0,19.750000,18.116667,2.100000,0.919346
1,360610002012,o_360610002012,1887.846508,1125.124,0,24.833333,19.083333,2.050000,0.908890
2,360610002021,o_360610002021,2267.091673,1076.668,0,23.033333,18.250000,2.316667,0.912373
3,360610002022,o_360610002022,2477.358670,1112.161,0,24.200000,18.850000,1.150000,0.910307
4,360610002023,o_360610002023,2339.396673,1148.973,0,24.283333,19.500000,1.066667,0.908525
5,360610002024,o_360610002024,2576.681020,847.335,0,20.983333,14.400000,4.366667,0.923853
6,360610002025,o_360610002025,3375.847529,1374.579,0,26.733333,23.233333,5.750000,0.883055
7,360610002026,o_360610002026,2823.592450,826.163,1,22.000000,14.050000,4.350000,0.702480
8,360610005001,o_360610005001,5193.439474,1461.789,1,32.366667,25.033333,5.600000,0.645979
9,360610005002,o_360610005002,4234.746951,1960.732,0,37.233333,33.233333,2.383333,0.840455


In [25]:
final_results.to_csv(OUTPUT_DIR / "NEW_YORK_CITY_ALL_results.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "NEW_YORK_CITY_ALL_weights.csv", index=True)

print("Saved NEW_YORK_CITY_ALL_results.csv")
print("Saved NEW_YORK_CITY_ALL_weights.csv")

Saved NEW_YORK_CITY_ALL_results.csv
Saved NEW_YORK_CITY_ALL_weights.csv
